# ============================================================
# **PROJECT TITLE:** Basic Document Q&A Bot using RAG
# **PLATFORM:** Google Colab
# **AUTHOR:** Niladri Dutta
# **PURPOSE:** Build an end-to-end Retrieval Augmented Generation system using PDFs / DOCX / TXT files.
# ============================================================

# ============================================================
# **STEP 1: INSTALL REQUIRED LIBRARIES**
# ============================================================
# **langchain               ->** Main framework for RAG pipeline
# **langchain-community     ->** Loaders / vectorstores
# **langchain-google-genai ->** Gemini integration
# **chromadb                ->** Vector Database
# **sentence-transformers   ->** Embedding model
# **pypdf                   ->** Read PDF files
# **python-docx             ->** Read DOCX files
# **gradio                  ->** Web UI
# ============================================================

In [1]:
!pip install -qU langchain langchain-community chromadb sentence-transformers pypdf python-docx gradio langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1

In [2]:
!pip install -qU langchain langchain-community langchain-text-splitters

# LangChain's Google Gemini integration package supports Gemini via API key. :contentReference[oaicite:0]{index=0}


# ============================================================
# **STEP 2: IMPORT LIBRARIES**
# ============================================================


In [6]:
import os
import shutil
import getpass
from google.colab import files

# PDF Loader
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Correct Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector DB
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# DOCX
import docx

# UI
import gradio as gr

# ============================================================
# **STEP 3: ENTER GEMINI API KEY**
# ============================================================
# Get free API key from:
# https://aistudio.google.com/app/apikey
# ============================================================

In [7]:
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API Key: ")

Enter API Key: ··········


# ============================================================
# **STEP 4: UPLOAD DOCUMENTS**
# ============================================================
# Upload 4 to 5 files:
# Example:
# ai.pdf
# ml.pdf
# python.txt
# data_science.docx
# deep_learning.pdf
# ============================================================

In [8]:
uploaded = files.upload()

Saving Digestive System.pdf to Digestive System.pdf
Saving Electricity_Basics.pdf to Electricity_Basics.pdf
Saving Photosynthesis.pdf to Photosynthesis.pdf
Saving Solar-System.pdf to Solar-System.pdf


# ============================================================
# **STEP 5: CREATE DATA FOLDER**
# ============================================================

In [9]:
os.makedirs("docs", exist_ok=True)

for file_name in uploaded.keys():
    shutil.move(file_name, f"docs/{file_name}")

print("Files moved to docs folder successfully.")

Files moved to docs folder successfully.


# ============================================================
# **STEP 6: LOAD ALL DOCUMENTS**
# ============================================================
# This function supports:
# PDF
# TXT
# DOCX
# ============================================================

In [10]:
documents = []

def load_docx(file_path):
    """
    Custom DOCX loader
    """
    doc = docx.Document(file_path)
    full_text = "\n".join([p.text for p in doc.paragraphs])

    from langchain.schema import Document
    return [Document(
        page_content=full_text,
        metadata={"source": os.path.basename(file_path)}
    )]

for file in os.listdir("docs"):

    path = os.path.join("docs", file)

    if file.endswith(".pdf"):
        loader = PyPDFLoader(path)
        docs = loader.load()
        documents.extend(docs)

    elif file.endswith(".txt"):
        loader = TextLoader(path)
        docs = loader.load()
        documents.extend(docs)

    elif file.endswith(".docx"):
        docs = load_docx(path)
        documents.extend(docs)

print("Total loaded pages/documents:", len(documents))

Total loaded pages/documents: 69


# ============================================================
# **STEP 7: TEXT CHUNKING**
# ============================================================
# Why chunking?
# Large documents are split into smaller meaningful pieces.
#
# chunk_size=1000:
# Each chunk approx 1000 chars
#
# overlap=200:
# Keeps context continuity between chunks
# ============================================================

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 135


# ============================================================
# **STEP 8: CREATE EMBEDDINGS**
# ============================================================
# Embeddings convert text into vectors.
# Similar meaning texts stay close mathematically.
#
# Using free HuggingFace model: all-MiniLM-L6-v2
# ============================================================

In [12]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_7122/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# ============================================================
# **STEP 9: CREATE VECTOR DATABASE**
# ============================================================
# Chroma stores embeddings locally.
# Persisted DB means no need to rebuild every time.
# ============================================================


In [13]:
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

db.persist()

print("Vector DB created successfully.")

Vector DB created successfully.


/tmp/ipykernel_7122/150531580.py:7: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


# ============================================================
# **STEP 10: LOAD GEMINI MODEL**
# ============================================================
# gemini-1.5-flash is fast and cost effective.
# ============================================================

In [14]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# ============================================================
# **STEP 11: MAIN QUESTION ANSWERING FUNCTION**
# ============================================================
# Flow:
# User Question
# -> Similarity Search
# -> Retrieve top 3 chunks
# -> Send chunks + question to Gemini
# -> Return grounded answer
# ============================================================

In [15]:
def ask_bot(question):

    # Search top relevant chunks
    docs = db.similarity_search(question, k=3)

    # If nothing found
    if not docs:
        return "No relevant information found."

    # Prepare context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Collect sources
    sources = []
    for d in docs:
        src = d.metadata.get("source", "Unknown File")
        page = d.metadata.get("page", "N/A")
        sources.append(f"{src} | Page: {page}")

    source_text = "\n".join(list(set(sources)))

    # Prompt engineering
    prompt = f"""
                   You are a helpful Document Q&A assistant.

                   Use ONLY the context below.
                   If answer not present, say:
                   'I could not find this in uploaded documents.'

                   Context:
                   {context}

                  Question:
                  {question}

                  Give a clear professional answer.
                  At the end show source citations.
                  """

    # Generate answer
    response = llm.invoke(prompt)

    final_answer = response.content + "\n\nSources:\n" + source_text

    return final_answer

# ============================================================
# **STEP 12: TEST IN NOTEBOOK**
# ============================================================

In [16]:
print(ask_bot("What is Resistance?"))

Resistance is defined as the opposition to the passage of an electric current. It is symbolized by 'R' and measured in Ohms (Ω).

Analogously, just as a smaller pipe offers greater resistance to water flow, a thinner wire presents greater resistance to electric current. A traditional incandescent light bulb, for example, is a high resistance wire. Resistance is also related to voltage and current through Ohm's Law (R = V / I).

Source: Slide 3, Slide 4

Sources:
docs/Electricity_Basics.pdf | Page: 2
docs/Electricity_Basics.pdf | Page: 3
docs/Electricity_Basics.pdf | Page: 1


# ============================================================
# **STEP 13: CREATE WEB UI USING GRADIO**
# ============================================================
# Easy web interface
# ============================================================

In [17]:
interface = gr.Interface(
    fn=ask_bot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask question from uploaded documents..."
    ),
    outputs="text",
    title="📄 Document Q&A Bot (RAG)",
    description="Upload files, ask questions, get answers with citations."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db5fe6254a18d128c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ============================================================
# **END OF PROJECT**
# ============================================================